In [1]:
import pandas as pd
import numpy as np
from causalexplain import GraphDiscovery
from graphviz import Digraph
from pathlib import Path
import re

def draw_graphviz_dag(adj, nodes,labels, out_path, engine="dot"):
    #guards to check for adjacency matrix dimension
    adj = np.asarray(adj)
    print("draw_graphviz_dag adj ndim:", adj.ndim, "shape:", adj.shape)

    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]
    nodes = list(nodes)[:n]
    
    # Restore original labels where possible
    labels = labels

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    g.attr(rankdir="TB")  # left-to-right; change to "TB" if you prefer top-down
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10"
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7"
    )

    # Add nodes with restored labels
    for clean_name, label in zip(nodes, labels):
        g.node(clean_name, label=label)

    # Add edges
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)



#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_ReX"
output_dir.mkdir(parents=True,exist_ok=True)

In [2]:
# Map from cleaned -> original for restoring labels in plots
def clean_and_encode_df(df: pd.DataFrame):
    """
    - Drop rows with NA.
    - Clean column names for algorithms (letters+digits, start with letter).
    - One-hot encode non-numeric columns.
    Returns:
      df_enc: encoded numeric DataFrame
      clean_to_orig: dict {clean_name: original_name}
    """
    df = df.dropna().copy()
    if df.empty:
        raise ValueError("Data frame is empty after dropna().")

    # 1) Clean base column names
    orig_cols = list(df.columns)
    clean_cols = []
    for c in orig_cols:
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)  # remove underscores, spaces, etc.
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        clean_cols.append(c2)
    df.columns = clean_cols
    clean_to_orig = dict(zip(clean_cols, orig_cols))

    # 2) One-hot encode non-numeric columns
    non_numeric = df.select_dtypes(exclude=["number"]).columns
    if len(non_numeric) > 0:
        df_enc = pd.get_dummies(df, columns=list(non_numeric), drop_first=False, dtype=float)
    else:
        df_enc = df.astype(float)

    return df_enc, clean_to_orig

In [3]:
def dot_to_adjacency(dot_path):
    G = nx.drawing.nx_pydot.read_dot(str(dot_path))
    nodes = list(G.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    p = len(nodes)
    adj = np.zeros((p, p), dtype=int)
    for u, v in G.edges():
        i = idx[u]
        j = idx[v]
        adj[i, j] = 1

    adj = np.asarray(adj)
    if adj.ndim == 1:              # safety
        adj = adj.reshape(1, 1)
    return adj, nodes

In [4]:
import networkx as nx
import time
def run_rex(df, experiment_name="rex_exp"):
    #Graph Discovery expects a file path for dataframe
    tmp_csv = "tmp_rex_input.csv"
    df = clean_name(df)
    df.to_csv(tmp_csv, index=False) #write df to a csv temporarily

    #init graph discovery instance
    start = time.time()
    gd = GraphDiscovery(experiment_name=experiment_name, model_type="rex",csv_filename= tmp_csv)

    gd.run(quiet=True)
    end=time.time()

    dot_path = output_dir / f"{experiment_name}.dot"
    gd.export_dag(str(dot_path))

    # Read DOT with networkx and build adjacency
    adj, nodes = dot_to_adjacency(dot_path)
    print("ReX adj ndim:", adj.ndim, "shape:", adj.shape)
    print(f"ReX took {(end - start)/60:.2f} minutes")

    return adj, nodes

In [5]:
def clean_name(df):
    #makes column names ReX friendly i.e start with letter, contain only letters and numbers
    new_cols = []
    for c in df.columns:
        # remove underscores and other non-alphanumeric
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)
        # if it doesn't start with a letter, prefix with 'X'
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        new_cols.append(c2)
    df2 = df.copy()
    df2.columns = new_cols
    return df2

def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc

In [6]:
csv_path = nij_root/"NIJ_lean_compact_onehot.csv"
df = pd.read_csv(csv_path)
df_num = encode_mixed_df(df)

orig_labels = df.columns

adj, nodes = run_rex(df_num)

#save recovered adjacency matrix to a csv
adj_df = pd.DataFrame(adj, index=df.columns, columns=df.columns)
adj_out_path = output_dir / "NIJ_graph_ReX_adj.csv"
adj_df.to_csv(adj_out_path, index=True)

out_path = output_dir / "NIJ_graph_ReX"
draw_graphviz_dag(adj, nodes, orig_labels, out_path)

G = nx.DiGraph()
#add nodes
G.add_nodes_from(nodes)

#add directed edges, adj[i,j] = 1 => i -> j
for i, src in enumerate(nodes):
    for j, tgt in enumerate(nodes):
        if adj[i, j] == 1:
            G.add_edge(src, tgt)

Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
ReX adj ndim: 2 shape: (21, 21)
ReX took 72.81 minutes
draw_graphviz_dag adj ndim: 2 shape: (21, 21)


In [5]:
import networkx as nx
adj_out_path = output_dir / "NIJ_graph_ReX_adj.csv"
df = pd.read_csv(adj_out_path, index_col=0)

nodes = list(df.columns)
adj = df.to_numpy()

G = nx.DiGraph()
G.add_nodes_from(nodes)

for i, src in enumerate(nodes):
    for j, tgt in enumerate(nodes):
        if adj[i, j] == 1:
            G.add_edge(src, tgt)

In [7]:
import networkx as nx
from collections import Counter

AGE_PREFIX = "Age_at_Release"          # prefix of one-hot age bucket columns
AGE_MERGED = "Age_at_Release"           # name of merged age node
TARGET     = "Recidivism_Within_3years" #target node
#G assumed to be an nx.DiGraph()

#Merge age buckets
age_nodes = [n for n in G.nodes if isinstance(n, str) and n.startswith(AGE_PREFIX)]
G_merged = nx.DiGraph()

# copy all nodes except age buckets
for n in G.nodes:
    if n not in age_nodes:
        G_merged.add_node(n)

# add merged age node if any buckets exist
if age_nodes:
    G_merged.add_node(AGE_MERGED)

age_out = Counter()   #counts outgoing edges from age buckets Age -> X (bucket -> non-age)
age_in  = Counter()   #counts incoming edges ".. "(non-age -> bucket)

for u, v, data in G.edges(data=True):

    #ignore edges between age buckets
    if u in age_nodes and v in age_nodes:
        continue

    #edges not affecting age buckets are just copied over
    if u not in age_nodes and v not in age_nodes:
        G_merged.add_edge(u, v, **data)
        continue

    #bucket -> non-age variable
    if u in age_nodes and v not in age_nodes:
        age_out[v] += 1

    #non-age variable -> bucket
    if v in age_nodes and u not in age_nodes:
        age_in[u] += 1

#decide majority direction for each neighbour of an age bucket
for x, cnt_out in age_out.items():
    cnt_in = age_in.get(x, 0)

    if cnt_out > cnt_in:
        # majority oriented as Age bucket -> X
        G_merged.add_edge(AGE_MERGED, x)
    elif cnt_in > cnt_out:
        # majority X -> Age bucket
        G_merged.add_edge(x, AGE_MERGED)
    else:
        #tie: drop, dont add edge to age merged
        pass

# add edges X -> Age for neighbours that only ever had incoming-to-bucket edges
for x, cnt_in in age_in.items():
    if x in age_out:
        continue    # already handled above
    G_merged.add_edge(x, AGE_MERGED)

if TARGET not in G_merged:
    raise ValueError(f"Target node '{TARGET}' not found in graph; "
                     f"available nodes include: {list(G_merged.nodes)[:10]} ...")

parents  = set(G_merged.predecessors(TARGET))
markov_blanket_nodes = parents | {TARGET}

G_mb = G_merged.subgraph(markov_blanket_nodes).copy()

print("Original nodes:", len(G.nodes))
print("After age-merge:", len(G_merged.nodes))
print("Markov blanket nodes:", len(G_mb.nodes))

import networkx as nx
from networkx.drawing.nx_agraph import to_agraph  # needs pygraphviz

A = to_agraph(G_mb)
A.graph_attr.update(rankdir="TB")  # or "LR" for left‑to‑right
# render to PNG with Graphviz
A.draw("NIJ_graph_ReX_PRUNED.png", format="png", prog="dot")


Original nodes: 21
After age-merge: 15
Markov blanket nodes: 2


In [8]:
print("Original edges:", len(G.edges()))
print("After age-merge:", len(G_merged.edges()))
print("Markov blanket edges:", len(G_mb.edges()))


Original edges: 62
After age-merge: 31
Markov blanket edges: 1


In [9]:
parents  = set(G_merged.predecessors(TARGET))
children = set(G_merged.successors(TARGET))

spouses = set()
for c in children:
    for p in G_merged.predecessors(c):
        if p != TARGET:
            spouses.add(p)

mbfull_nodes = parents | children | spouses | {TARGET}

G_mbfull = G_merged.subgraph(mbfull_nodes).copy()
A = to_agraph(G_mbfull)
A.graph_attr.update(rankdir="TB")  # or "LR"
A.draw("NIJ_graph_ReX_MBFULL.png", format="png", prog="dot")

In [10]:
print("Original nodes:", len(G.nodes))
print("After age-merge:", len(G_merged.nodes))
print("Markov blanket nodes:", len(G_mbfull.nodes))
print("Original edges:", len(G.edges()))
print("After age-merge:", len(G_merged.edges()))
print("Markov blanket edges:", len(G_mbfull.edges()))

Original nodes: 21
After age-merge: 15
Markov blanket nodes: 8
Original edges: 62
After age-merge: 31
Markov blanket edges: 15
